<a href="https://colab.research.google.com/github/archipelagoing/Situationion/blob/main/v5_Decodability_MechDiagnostic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SituatiONION V5: Decodability Atlas

This notebook maps held-out linear decodability across situation variables, token readouts, and GPT-2 XL layers. It uses V4's frozen long-format data and hidden-state cache. Decodability is not causal necessity.

# V5- Initial idea.

**Central question: What specific components of the situation can be decoded at different depths?**

V5 would contain:

Layer-wise probes.
Agent decoding.
Recipient decoding.
Event decoding.
Causality decoding.
Temporal-relation decoding.
Polarity decoding.
Held-out-template evaluation.
Attention comparison.
PCA vs random-projection controls.
Stronger non-semantic controls if V4 identifies remaining concerns.

This is where you investigate whether the geometry from V4 actually corresponds to identifiable situation variables.

Your progression becomes:

$$ \text{V4: Where is information?} $$ $$ \downarrow $$ $$ \text{V5: What information is there?} $$

## Held-Out Linear Probes

Identity probes test only held-out templates with seen entities. Binary situation-variable probes test templates, entities, and combined holdouts. Each task reports linear, shuffled-label, random-projection, lexical, and majority controls.

# V5 — Decodability and Mechanistic Diagnostics

## Central Question

**Which situation variables are decodable from which token representations at different depths, and how does their decodability change across the network?**

V4 established that situation-sensitive geometry is distributed across GPT-2 XL rather than localized to a single privileged layer. It also showed that this geometry depends strongly on both the **situation dimension being manipulated** and the **token representation being measured**.

V5 therefore moves from measuring geometric separation to identifying the information represented by that geometry.

$$
\text{V4: Where is situation-sensitive information?}
$$

$$
\downarrow
$$

$$
\text{V5: What situation information is represented there?}
$$

## Primary Analyses

### 1. Layer × Readout Probing

Train layer-wise probes at multiple token representations rather than probing only one sentence-level representation.

Readouts should include:

* Changed token
* Event token
* Final token
* Agent/recipient token where applicable
* Mean pool as a global baseline
* Max pool as a negative/weak baseline where useful

The goal is to construct a **layer × readout × situation-variable** decodability map.

### 2. Situation-Variable Decoding

Probe separately for:

* Agent
* Recipient
* Agent/recipient role assignment
* Event/state
* Causality
* Polarity
* Temporal relation

These variables should be treated as separate prediction problems rather than collapsed into a single "situation" label.

### 3. Held-Out Generalization

Probe performance must be evaluated on held-out templates or structural families.

Random train/test sentence splits are insufficient because a probe could exploit lexical or template-specific regularities rather than situation information.

Where possible, evaluate:

* Held-out templates
* Held-out lexical items
* Held-out situation combinations

This asks whether the representation encodes a generalizable situation variable rather than memorizing the construction used to express it.

### 4. Probe Controls

For every probe, compare against appropriate controls:

* Majority/chance baseline
* Label-shuffled probe
* Lexical/surface-feature baseline
* Random-projection representation control
* Where appropriate, embedding-layer or early-layer baseline

Probe complexity should be kept deliberately limited, beginning with linear probes.

The purpose is to measure information that is readily available in the representation, not information that a powerful classifier can reconstruct.

### 5. V4-Guided Predictions

V5 should preregister several predictions based on V4.

**P1 — Token specificity**

Situation variables should generally be more decodable from semantically relevant token representations than from global pooled representations.

**P2 — Changed-token early availability**

Variables directly expressed by the manipulated token may already be decodable in very early layers.

This would distinguish **local lexical availability** from later contextual integration.

**P3 — Event-token contextual development**

Event/state information should show increasing decodability at the Event-token representation through early and middle layers, consistent with the depth-dependent trajectory observed in V4.

**P4 — Final-token integration**

Multiple situation variables may become decodable from the Final-token representation, consistent with contextual information becoming distributed beyond its original token location.

**P5 — Dimension-specific trajectories**

Agent/recipient, event/state, causality, polarity, and temporal relations should not exhibit identical layer-wise decodability curves.

**P6 — Temporal uncertainty**

Because V4 found weak temporal separation, temporal decoding is an important diagnostic rather than an expected positive result.

Failure to decode temporal relations would itself constrain the broader situation-model hypothesis.

### 6. Geometry–Decodability Correspondence

Compare V4 geometric separation with V5 probe performance.

For each:

$$
(\text{dimension},\text{readout},\text{layer})
$$

ask whether stronger counterfactual-vs-paraphrase separation predicts greater decodability of the corresponding situation variable.

This directly tests whether the geometry identified in V4 corresponds to identifiable semantic information.

## Secondary Mechanistic Diagnostics

### Attention Analysis

Compare attention patterns only as a secondary diagnostic.

Attention should not by itself be interpreted as evidence that a component stores or causally uses situation information.

Ask whether changes in decodability coincide with changes in information flow between situation-relevant token positions.

### PCA vs Random Projection

Retain the random-projection comparison from V4 as a visualization control.

PCA should be treated as descriptive visualization rather than primary evidence.

The primary quantitative evidence in V5 should come from held-out decoding performance and its controls.

## Interpretation Boundary

V5 tests **decodability**, not causal necessity.

Successful decoding establishes that information about a situation variable is available in a representation under the probe's assumptions. It does not establish that GPT-2 XL itself uses that information to produce its predictions.

That distinction is reserved for V6:

$$
\text{V4: Where is information?}
$$

$$
\downarrow
$$

$$
\text{V5: What information is there?}
$$

$$
\downarrow
$$

$$
\text{V6: Does the model causally use it?}
$$

## V5 Goal

Produce a controlled map of:

$$
\boxed{
\text{Situation Variable}
\times
\text{Token Readout}
\times
\text{Layer}
\rightarrow
\text{Decodability}
}
$$

The central V5 result should therefore not be a single "best layer."

It should be a **decodability atlas showing what information is available where, and how that availability changes through the network.**


# 1. Setup and inputs
   Load data/v5_probe_examples.csv, define output paths, seeds, all 48 layers, the target variables, and the six token readouts. Load the cached V4 GPT-2 XL hidden states from:
results/layer_curves/gpt2xl_hidden_states.pt


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/Fall26/Coding/SituationionArtifacts"
)
DATA_PATH = ARTIFACT_DIR / "v5_probe_examples.csv"
CACHE_PATH = ARTIFACT_DIR / "gpt2xl_hidden_states.pt"

if DATA_PATH.exists():
    print(f"Data path found: {DATA_PATH}")
else:
    print(f"Data path not found: {DATA_PATH}")

Data path found: /content/drive/MyDrive/Fall26/Coding/SituationionArtifacts/v5_probe_examples.csv


In [3]:
!ls -lh /content/drive/MyDrive/Fall26/Coding/SituationionArtifacts


total 2.9G
-rw------- 1 root root 124K Sep  4 00:13 candidate_triples.csv
-rw------- 1 root root 2.9G Sep  4 00:40 gpt2xl_hidden_states.pt
-rw------- 1 root root 124K Sep  4 00:13 matched_triples.csv
drwx------ 2 root root 4.0K Sep  4 18:25 v5_decodability
-rw------- 1 root root 223K Sep  4 00:13 v5_probe_examples.csv


In [4]:
import re
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.random_projection import GaussianRandomProjection
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()

In [5]:
from joblib import Parallel, delayed

def probe_score(x_train, y_train, x_test, y_test):
    if len(set(y_train)) < 2 or len(set(y_test)) < 2: return np.nan
    probe = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED)

    # Scale the data to prevent convergence warnings
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)

    return balanced_accuracy_score(y_test, probe.fit(x_train_scaled, y_train).predict(x_test_scaled))

def eligible_splits(task, data):
    return ('test_template',) if task in {'agent_identity', 'recipient_identity'} else tuple(split for split in ('test_template', 'test_entity', 'test_both') if (data.split == split).any())

In [6]:
# DATA_PATH = ROOT / 'data' / 'v5_probe_examples.csv'
# CACHE_PATH = ROOT / 'results' / 'layer_curves' / 'gpt2xl_hidden_states.pt'

OUTPUT_DIR = ARTIFACT_DIR / "v5_decodability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED, LAYERS = 7, range(48)
rng = np.random.default_rng(SEED)
TASKS = {'agent_identity': ('semantic_agent', 'agent_identity_eligible'), 'recipient_identity': ('semantic_recipient', 'recipient_identity_eligible'), 'role_assignment': ('role_assignment', 'role_assignment_eligible'), 'event_state': ('event_state', 'event_state_eligible'), 'cause_holder_role': ('cause_holder_role', 'cause_holder_role_eligible'), 'polarity': ('polarity', 'polarity_eligible'), 'temporal_relation': ('temporal_relation', 'temporal_relation_eligible')}
READOUTS = ('changed_token', 'event_token', 'final_token', 'agent_recipient', 'mean_pool', 'max_pool')
if not DATA_PATH.exists(): raise FileNotFoundError('Run python3 structural_controls.py first.')
if not CACHE_PATH.exists(): raise FileNotFoundError('Copy gpt2xl_hidden_states.pt from V4 into results/layer_curves/.')
probe_examples = pd.read_csv(DATA_PATH)
cache = torch.load(CACHE_PATH, map_location='cpu', weights_only=False)
tokenizer, RUNS = cache['tokenizer'], cache['runs']
print(f'Loaded {len(probe_examples)} sentences and {len(RUNS)} cached triples.')


Loaded 900 sentences and 300 cached triples.


# 2. Readout extraction
   Write functions that return a representation for one sentence at one layer:
   - Changed-token hidden state
   - Event-token hidden state
   - Final-token hidden state
   - Ordered agent/recipient representation
   - Mean pool
   - Max pool
The long-format CSV supplies triple_id, variant, semantic role labels, and eligibility flags; the hidden-state cache supplies the actual tensors.


In [7]:
EVENT = {'agent_recipient': ('gave', 'gave', 'gave'), 'cause': ('called', 'called', 'called'), 'temporal': ('cleaned', 'cleaned', 'cleaned'), 'polarity': ('repair', 'mend', 'fix'), 'event_state': ('carried', 'transported', 'dropped')}
CHANGED = {'temporal': ('after', 'once', 'before'), 'polarity': ('repair', 'mend', 'not'), 'event_state': ('carried', 'transported', 'dropped')}
def token_position(text, surface):
    matches = list(re.finditer(r'(?<![a-zA-Z])' + re.escape(surface) + r'(?![a-zA-Z])', text, re.I))
    if not matches: return None # Return None if surface is not found
    return len(tokenizer.encode(text[:matches[-1].end()], add_special_tokens=False)) - 1
def readout_vector(row, layer, readout):
    run = RUNS[row.triple_id][row.variant]; hidden = run['hidden_states'][layer + 1]
    index = ('base', 'paraphrase', 'counterfactual').index(row.variant)
    if readout == 'mean_pool': return hidden.mean(0).numpy()
    if readout == 'max_pool': return hidden.max(0).values.numpy()
    if readout == 'final_token': return hidden[-1].numpy()
    if readout == 'event_token':
        pos = token_position(run['text'], EVENT[row.manipulation][index])
        if pos is None: return None # Handle case where token is not found
        return hidden[pos].numpy()
    if readout == 'changed_token':
        surface = row.semantic_recipient if row.manipulation == 'agent_recipient' and row.variant == 'counterfactual' else (row.semantic_agent if row.manipulation in {'agent_recipient', 'cause'} else CHANGED[row.manipulation][index])
        pos = token_position(run['text'], surface)
        if pos is None: return None # Handle case where token is not found
        return hidden[pos].numpy()
    # For 'agent_recipient' readout
    agent_pos = token_position(run['text'], row.semantic_agent)
    recipient_pos = token_position(run['text'], row.semantic_recipient)

    if agent_pos is None or recipient_pos is None:
        return None # If either agent or recipient token is not found, return None for the combined vector

    return torch.cat((hidden[agent_pos], hidden[recipient_pos])).numpy()
test_vector_result = readout_vector(probe_examples.iloc[0], 0, 'final_token')
if test_vector_result is not None:
    print('Readout extraction ready:', test_vector_result.shape)
else:
    print('Readout extraction ready: Test vector could not be extracted.')


Readout extraction ready: (1600,)


# 3. Held-out linear probes
   For each valid combination of:
task × readout × layer × evaluation split
fit a class-balanced logistic-regression probe on split == "dev" and evaluate separately on:
- test_template
- test_entity
- test_both
Use balanced accuracy. Identity tasks should use only held-out templates with known entity labels. Do not run identity classification on unseen entity names.


In [ ]:
# =============================================================================
# V3: CACHED LINEAR PROBES + PROGRESS TRACKING
# Precompute readout vectors once to avoid redundant calculations,
# then reuse cached representations across tasks, splits, readouts, and layers.
# Includes tqdm progress bars for both vector caching and probe evaluation.
# =============================================================================
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score


def probe_score(x_train, y_train, x_test, y_test):
    if len(set(y_train)) < 2 or len(set(y_test)) < 2:
        return np.nan

    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)

    probe = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,   # helps for some multiclass solvers
    )

    probe.fit(x_train_scaled, y_train)

    return balanced_accuracy_score(
        y_test,
        probe.predict(x_test_scaled)
    )


def eligible_splits(task, data):
    if task in {"agent_identity", "recipient_identity"}:
        return ("test_template",)

    return tuple(
        split
        for split in ("test_template", "test_entity", "test_both")
        if (data.split == split).any()
    )


# ---------------------------------------------------------
# 1. CACHE READOUT VECTORS
# ---------------------------------------------------------

vector_cache = {}

# Collect all rows that could possibly be used
relevant_indices = probe_examples.index.tolist()

total_cache_jobs = (
    len(relevant_indices)
    * len(READOUTS)
    * len(LAYERS)
)

print(f"Precomputing {total_cache_jobs:,} readout vectors...")

with tqdm(
    total=total_cache_jobs,
    desc="Caching readout vectors",
    unit="vec"
) as pbar:

    for idx, row in probe_examples.iterrows():

        for readout in READOUTS:

            for layer in LAYERS:

                key = (idx, layer, readout)

                vector_cache[key] = readout_vector(
                    row,
                    layer,
                    readout
                )

                pbar.update(1)


print(f"Cached {len(vector_cache):,} vectors.")


# ---------------------------------------------------------
# 2. COUNT PROBE JOBS
# ---------------------------------------------------------

total_probe_jobs = 0

for task, (label, flag) in TASKS.items():
    data = probe_examples[probe_examples[flag]]

    total_probe_jobs += (
        len(eligible_splits(task, data))
        * len(READOUTS)
        * len(LAYERS)
    )


print(f"Running {total_probe_jobs:,} linear probes...")


# ---------------------------------------------------------
# 3. RUN PROBES
# ---------------------------------------------------------

linear_rows = []

with tqdm(
    total=total_probe_jobs,
    desc="Linear probes",
    unit="probe"
) as pbar:

    for task, (label, flag) in TASKS.items():

        data = probe_examples[probe_examples[flag]].copy()

        train = data.query("split == 'dev'")

        for test_split in eligible_splits(task, data):

            test = data.query("split == @test_split")

            for readout in READOUTS:

                for layer in LAYERS:

                    # -----------------------------------
                    # TRAIN
                    # -----------------------------------

                    train_vectors = []
                    train_labels = []

                    for idx, y_val in zip(
                        train.index,
                        train[label].to_numpy()
                    ):
                        vec = vector_cache[(idx, layer, readout)]

                        if vec is not None:
                            train_vectors.append(vec)
                            train_labels.append(y_val)

                    # -----------------------------------
                    # TEST
                    # -----------------------------------

                    test_vectors = []
                    test_labels = []

                    for idx, y_val in zip(
                        test.index,
                        test[label].to_numpy()
                    ):
                        vec = vector_cache[(idx, layer, readout)]

                        if vec is not None:
                            test_vectors.append(vec)
                            test_labels.append(y_val)

                    # -----------------------------------
                    # SCORE
                    # -----------------------------------

                    if (
                        len(train_vectors) == 0
                        or len(test_vectors) == 0
                        or len(set(train_labels)) < 2
                        or len(set(test_labels)) < 2
                    ):
                        score = np.nan

                    else:
                        x_train = np.stack(train_vectors)
                        y_train = np.asarray(train_labels)

                        x_test = np.stack(test_vectors)
                        y_test = np.asarray(test_labels)

                        score = probe_score(
                            x_train,
                            y_train,
                            x_test,
                            y_test
                        )

                    linear_rows.append({
                        "task": task,
                        "readout": readout,
                        "layer": layer,
                        "test_split": test_split,
                        "control": "linear",
                        "score": score,
                    })

                    # Show current job in tqdm
                    pbar.set_postfix(
                        task=task,
                        split=test_split,
                        readout=readout,
                        layer=layer,
                        score=(
                            f"{score:.3f}"
                            if np.isfinite(score)
                            else "nan"
                        )
                    )

                    pbar.update(1)


linear_atlas = pd.DataFrame(linear_rows)

print(
    f"Finished {len(linear_atlas):,} held-out linear probes."
)

Precomputing 259,200 readout vectors...


Caching readout vectors:   0%|          | 0/259200 [00:00<?, ?vec/s]

Cached 259,200 vectors.
Running 4,896 linear probes...


Linear probes:   0%|          | 0/4896 [00:00<?, ?probe/s]

In [ ]:
# # =============================================================================
# # V3.4:
#       *----|CACHED + PARALLEL LINEAR PROBES + PROGRESS TRACKING |----*
# # Precompute readout vectors once, then run independent probe combinations
# # in parallel across CPU cores while tracking completed jobs with tqdm.
# # =============================================================================

# from tqdm.auto import tqdm
# from joblib import Parallel, delayed
# import numpy as np
# import pandas as pd

# from sklearn.linear_model import LogisticRegression
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import balanced_accuracy_score


# def probe_score(x_train, y_train, x_test, y_test):
#     if len(set(y_train)) < 2 or len(set(y_test)) < 2:
#         return np.nan

#     scaler = StandardScaler()

#     x_train_scaled = scaler.fit_transform(x_train)
#     x_test_scaled = scaler.transform(x_test)

#     probe = LogisticRegression(
#         max_iter=2000,
#         class_weight="balanced",
#         random_state=SEED
#     )

#     probe.fit(x_train_scaled, y_train)

#     return balanced_accuracy_score(
#         y_test,
#         probe.predict(x_test_scaled)
#     )


# def eligible_splits(task, data):
#     if task in {"agent_identity", "recipient_identity"}:
#         return ("test_template",)

#     return tuple(
#         split
#         for split in ("test_template", "test_entity", "test_both")
#         if (data.split == split).any()
#     )


# # =============================================================================
# # 1. CACHE READOUT VECTORS ONCE
# # =============================================================================

# vector_cache = {}

# total_cache_jobs = (
#     len(probe_examples)
#     * len(READOUTS)
#     * len(LAYERS)
# )

# print(f"Precomputing {total_cache_jobs:,} readout vectors...")

# with tqdm(
#     total=total_cache_jobs,
#     desc="Caching readout vectors",
#     unit="vec"
# ) as pbar:

#     for idx, row in probe_examples.iterrows():

#         for readout in READOUTS:

#             for layer in LAYERS:

#                 vector_cache[(idx, layer, readout)] = readout_vector(
#                     row,
#                     layer,
#                     readout
#                 )

#                 pbar.update(1)


# print(f"Cached {len(vector_cache):,} vectors.")


# # =============================================================================
# # 2. BUILD ALL PROBE JOBS
# # =============================================================================

# probe_jobs = []

# for task, (label, flag) in TASKS.items():

#     data = probe_examples[probe_examples[flag]].copy()

#     train = data.query("split == 'dev'")

#     for test_split in eligible_splits(task, data):

#         test = data.query("split == @test_split")

#         for readout in READOUTS:

#             for layer in LAYERS:

#                 probe_jobs.append({
#                     "task": task,
#                     "label": label,
#                     "test_split": test_split,
#                     "readout": readout,
#                     "layer": layer,
#                     "train_indices": train.index.to_numpy(),
#                     "test_indices": test.index.to_numpy(),
#                 })


# print(f"Prepared {len(probe_jobs):,} linear probe jobs.")


# # =============================================================================
# # 3. FUNCTION FOR ONE PROBE JOB
# # =============================================================================

# def run_probe_job(job):

#     task = job["task"]
#     label = job["label"]
#     test_split = job["test_split"]
#     readout = job["readout"]
#     layer = job["layer"]

#     train_indices = job["train_indices"]
#     test_indices = job["test_indices"]

#     # -------------------------------------------------------------------------
#     # TRAIN DATA
#     # -------------------------------------------------------------------------

#     train_vectors = []
#     train_labels = []

#     for idx in train_indices:

#         vec = vector_cache[(idx, layer, readout)]

#         if vec is not None:
#             train_vectors.append(vec)
#             train_labels.append(
#                 probe_examples.at[idx, label]
#             )

#     # -------------------------------------------------------------------------
#     # TEST DATA
#     # -------------------------------------------------------------------------

#     test_vectors = []
#     test_labels = []

#     for idx in test_indices:

#         vec = vector_cache[(idx, layer, readout)]

#         if vec is not None:
#             test_vectors.append(vec)
#             test_labels.append(
#                 probe_examples.at[idx, label]
#             )

#     # -------------------------------------------------------------------------
#     # SCORE
#     # -------------------------------------------------------------------------

#     if (
#         len(train_vectors) == 0
#         or len(test_vectors) == 0
#         or len(set(train_labels)) < 2
#         or len(set(test_labels)) < 2
#     ):
#         score = np.nan

#     else:

#         x_train = np.stack(train_vectors)
#         y_train = np.asarray(train_labels)

#         x_test = np.stack(test_vectors)
#         y_test = np.asarray(test_labels)

#         score = probe_score(
#             x_train,
#             y_train,
#             x_test,
#             y_test
#         )

#     return {
#         "task": task,
#         "readout": readout,
#         "layer": layer,
#         "test_split": test_split,
#         "control": "linear",
#         "score": score,
#     }


# # =============================================================================
# # 4. RUN PROBES IN PARALLEL
# # =============================================================================

# print(
#     f"Running {len(probe_jobs):,} probes "
#     f"across available CPU cores..."
# )

# results = Parallel(
#     n_jobs=-1,
#     backend="loky"
# )(
#     delayed(run_probe_job)(job)
#     for job in tqdm(
#         probe_jobs,
#         desc="Submitting probe jobs",
#         unit="probe"
#     )
# )


# # =============================================================================
# # 5. COLLECT RESULTS
# # =============================================================================

# linear_atlas = pd.DataFrame(results)

# print(
#     f"Finished {len(linear_atlas):,} held-out linear probes."
# )

In [ ]:
# #### V2: PARALLELL VERSION!!!#########################
#-----------------------------------------------
#-----------------------------------------------
# from joblib import Parallel, delayed

# def process_probe_combination(task, label, flag, test_split, readout, layer, train_data, test_data):
#     data = probe_examples[probe_examples[flag]].copy()
#     train = train_data
#     test = test_data

#     # Generate vectors and filter out None values
#     x_train_vectors = [readout_vector(row, layer, readout) for row in train.itertuples(index=False)]
#     y_train_filtered = [y_val for i, y_val in enumerate(train[label].to_numpy()) if x_train_vectors[i] is not None]
#     x_train_filtered = [vec for vec in x_train_vectors if vec is not None]

#     x_test_vectors = [readout_vector(row, layer, readout) for row in test.itertuples(index=False)]
#     y_test_filtered = [y_val for i, y_val in enumerate(test[label].to_numpy()) if x_test_vectors[i] is not None]
#     x_test_filtered = [vec for vec in x_test_vectors if vec is not None]

#     if not x_train_filtered or not x_test_filtered or len(set(y_train_filtered)) < 2 or len(set(y_test_filtered)) < 2:
#         score = np.nan
#     else:
#         x_train = np.stack(x_train_filtered)
#         y_train = np.array(y_train_filtered)
#         x_test = np.stack(x_test_filtered)
#         y_test = np.array(y_test_filtered)
#         score = probe_score(x_train, y_train, x_test, y_test)

#     return {'task': task, 'readout': readout, 'layer': layer, 'test_split': test_split, 'control': 'linear', 'score': score}

# linear_results = []
# for task, (label, flag) in TASKS.items():
#     data = probe_examples[probe_examples[flag]].copy()
#     train = data.query("split == 'dev'")
#     for test_split in eligible_splits(task, data):
#         test = data.query('split == @test_split')
#         # Parallelize the inner loops for readouts and layers
#         results_for_task_split = Parallel(n_jobs=-1)(delayed(process_probe_combination)(
#             task, label, flag, test_split, readout, layer, train, test
#         ) for readout in READOUTS for layer in LAYERS)
#         linear_results.extend(results_for_task_split)

# linear_atlas = pd.DataFrame(linear_results)
# print(f'Finished {len(linear_atlas)} held-out linear probes.')

In [11]:
# V1: NO PARALLEL##############################
######-----------------------------------------
#----------------------------------------------------
#--------------------------------------------------
# def probe_score(x_train, y_train, x_test, y_test):
#     if len(set(y_train)) < 2 or len(set(y_test)) < 2: return np.nan
#     probe = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED)

#     # Scale the data to prevent convergence warnings
#     scaler = StandardScaler()
#     x_train_scaled = scaler.fit_transform(x_train)
#     x_test_scaled = scaler.transform(x_test)

#     return balanced_accuracy_score(y_test, probe.fit(x_train_scaled, y_train).
# def eligible_splits(task, data):
#     return ('test_template',) if task in {'agent_identity', 'recipient_identity'} else tuple(split for split in ('test_template', 'test_entity', 'test_both') if (data.split == split).any())

# linear_rows = []
# for task, (label, flag) in TASKS.items():
#     data = probe_examples[probe_examples[flag]].copy(); train = data.query("split == 'dev'")
#     for test_split in eligible_splits(task, data):
#         test = data.query('split == @test_split')
#         for readout in READOUTS:
#             for layer in LAYERS:
#                 # Generate vectors and filter out None values
#                 x_train_vectors = [readout_vector(row, layer, readout) for row in train.itertuples(index=False)]
#                 y_train_filtered = [y_val for i, y_val in enumerate(train[label].to_numpy()) if x_train_vectors[i] is not None]
#                 x_train_filtered = [vec for vec in x_train_vectors if vec is not None]

#                 x_test_vectors = [readout_vector(row, layer, readout) for row in test.itertuples(index=False)]
#                 y_test_filtered = [y_val for i, y_val in enumerate(test[label].to_numpy()) if x_test_vectors[i] is not None]
#                 x_test_filtered = [vec for vec in x_test_vectors if vec is not None]

#                 if not x_train_filtered or not x_test_filtered or len(set(y_train_filtered)) < 2 or len(set(y_test_filtered)) < 2: # Check if enough valid data points remain for training/testing
#                     score = np.nan
#                 else:
#                     x_train = np.stack(x_train_filtered)
#                     y_train = np.array(y_train_filtered)
#                     x_test = np.stack(x_test_filtered)
#                     y_test = np.array(y_test_filtered)
#                     score = probe_score(x_train, y_train, x_test, y_test)

#                 linear_rows.append({'task': task, 'readout': readout, 'layer': layer, 'test_split': test_split, 'control': 'linear', 'score': score})
# linear_atlas = pd.DataFrame(linear_rows)
# print(f'Finished {len(linear_atlas)} held-out linear probes.')

KeyboardInterrupt: 

# 4. Controls
   For every probe result, include:
   - Majority-class baseline
   - Label-shuffled logistic probe
   - TF-IDF word/bigram logistic probe
   - Random-projected hidden-state probe
   - Layer 0 as the early/embedding baseline
Store all scores in one CSV, for example:
results/v5_decodability/decodability_atlas.csv
with columns such as task, readout, layer, test_split, control, and score.


In [ ]:
# =============================================================================
# V4.3:
# *----|REUSE VECTOR CACHE + PARALLEL CONTROLS + PROGRESS TRACKING|----*
#
# Improvements over V2:
#   - Reuses the existing vector_cache built during linear probing
#   - Avoids recomputing readout_vector() entirely
#   - Builds train/test matrices from cached representations
#   - Runs shuffled-label and random-projection controls in parallel
#   - Uses deterministic shuffled-label seeds for reproducibility
#   - Tracks progress with tqdm
#   - Computes majority baseline using the same balanced-accuracy metric
# =============================================================================

from tqdm.auto import tqdm
from joblib import Parallel, delayed
from scipy import sparse
import hashlib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.random_projection import GaussianRandomProjection


# =============================================================================
# 1. PROBE SCORING
# =============================================================================

def probe_score(x_train, y_train, x_test, y_test):

    if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
        return np.nan

    # TF-IDF matrices are sparse and should not be mean-centered
    if sparse.issparse(x_train):
        scaler = StandardScaler(with_mean=False)
    else:
        scaler = StandardScaler()

    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)

    probe = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=SEED
    )

    probe.fit(x_train_scaled, y_train)

    return balanced_accuracy_score(
        y_test,
        probe.predict(x_test_scaled)
    )


# =============================================================================
# 2. DETERMINISTIC SEED FOR SHUFFLED-LABEL CONTROLS
# =============================================================================

def control_seed(task, test_split, readout, layer):

    key = f"{SEED}|{task}|{test_split}|{readout}|{layer}"

    digest = hashlib.sha256(
        key.encode("utf-8")
    ).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little"
    )


# =============================================================================
# 3. BUILD TRAIN/TEST MATRICES FROM EXISTING VECTOR CACHE
# =============================================================================

def cached_matrix(indices, labels, layer, readout):

    vectors = []
    filtered_labels = []

    for idx, label in zip(indices, labels):

        vec = vector_cache.get(
            (idx, layer, readout)
        )

        if vec is not None:
            vectors.append(vec)
            filtered_labels.append(label)

    if len(vectors) == 0:
        return None, None

    return (
        np.stack(vectors),
        np.asarray(filtered_labels)
    )


# =============================================================================
# 4. RUN ONE HIDDEN-STATE CONTROL JOB
# =============================================================================

def process_control_combination(
    task,
    test_split,
    readout,
    layer,
    train_indices,
    test_indices,
    y_train_raw,
    y_test_raw
):

    x_train, y_train = cached_matrix(
        train_indices,
        y_train_raw,
        layer,
        readout
    )

    x_test, y_test = cached_matrix(
        test_indices,
        y_test_raw,
        layer,
        readout
    )

    results = []

    if (
        x_train is None
        or x_test is None
        or len(np.unique(y_train)) < 2
        or len(np.unique(y_test)) < 2
    ):

        return [
            {
                "task": task,
                "readout": readout,
                "layer": layer,
                "test_split": test_split,
                "control": "shuffled_labels",
                "score": np.nan
            },
            {
                "task": task,
                "readout": readout,
                "layer": layer,
                "test_split": test_split,
                "control": "random_projection",
                "score": np.nan
            }
        ]


    # -------------------------------------------------------------------------
    # SHUFFLED LABEL CONTROL
    # -------------------------------------------------------------------------

    local_rng = np.random.default_rng(
        control_seed(
            task,
            test_split,
            readout,
            layer
        )
    )

    shuffled_y = local_rng.permutation(
        y_train
    )

    shuffled_score = probe_score(
        x_train,
        shuffled_y,
        x_test,
        y_test
    )

    results.append({
        "task": task,
        "readout": readout,
        "layer": layer,
        "test_split": test_split,
        "control": "shuffled_labels",
        "score": shuffled_score
    })


    # -------------------------------------------------------------------------
    # RANDOM PROJECTION CONTROL
    # -------------------------------------------------------------------------

    rp = GaussianRandomProjection(
        n_components=min(
            128,
            x_train.shape[1]
        ),
        random_state=SEED
    )

    x_train_rp = rp.fit_transform(
        x_train
    )

    x_test_rp = rp.transform(
        x_test
    )

    rp_score = probe_score(
        x_train_rp,
        y_train,
        x_test_rp,
        y_test
    )

    results.append({
        "task": task,
        "readout": readout,
        "layer": layer,
        "test_split": test_split,
        "control": "random_projection",
        "score": rp_score
    })

    return results


# =============================================================================
# 5. NON-LAYER-SPECIFIC CONTROLS
# =============================================================================

control_rows = []
control_jobs = []

for task, (label, flag) in TASKS.items():

    data = probe_examples[
        probe_examples[flag]
    ].copy()

    train = data.query(
        "split == 'dev'"
    )

    for test_split in eligible_splits(
        task,
        data
    ):

        test = data.query(
            "split == @test_split"
        )

        y_train = train[label].to_numpy()
        y_test = test[label].to_numpy()


        # ---------------------------------------------------------------------
        # MAJORITY CLASS BASELINE
        # ---------------------------------------------------------------------

        majority_class = train[label].mode().iloc[0]

        majority_predictions = np.full(
            len(y_test),
            majority_class,
            dtype=object
        )

        majority_score = balanced_accuracy_score(
            y_test,
            majority_predictions
        )

        control_rows.append({
            "task": task,
            "readout": "majority",
            "layer": -1,
            "test_split": test_split,
            "control": "majority",
            "score": majority_score
        })


        # ---------------------------------------------------------------------
        # TF-IDF WORD/BIGRAM CONTROL
        # ---------------------------------------------------------------------

        try:

            tfidf = TfidfVectorizer(
                ngram_range=(1, 2)
            )

            x_train_tfidf = tfidf.fit_transform(
                train.text
            )

            x_test_tfidf = tfidf.transform(
                test.text
            )

            tfidf_score = probe_score(
                x_train_tfidf,
                y_train,
                x_test_tfidf,
                y_test
            )

        except ValueError:
            tfidf_score = np.nan

        control_rows.append({
            "task": task,
            "readout": "lexical_tfidf",
            "layer": -1,
            "test_split": test_split,
            "control": "lexical",
            "score": tfidf_score
        })


        # ---------------------------------------------------------------------
        # PREPARE PARALLEL HIDDEN-STATE JOBS
        # ---------------------------------------------------------------------

        train_indices = train.index.to_numpy()
        test_indices = test.index.to_numpy()

        for readout in READOUTS:

            for layer in LAYERS:

                control_jobs.append(
                    (
                        task,
                        test_split,
                        readout,
                        layer,
                        train_indices,
                        test_indices,
                        y_train,
                        y_test
                    )
                )


print(
    f"Prepared {len(control_jobs):,} hidden-state control jobs."
)


# =============================================================================
# 6. PARALLEL HIDDEN-STATE CONTROLS
# =============================================================================

parallel_results = Parallel(
    n_jobs=-1,
    backend="threading"
)(
    delayed(process_control_combination)(*job)

    for job in tqdm(
        control_jobs,
        desc="Running controls",
        unit="probe"
    )
)


# =============================================================================
# 7. FLATTEN RESULTS
# =============================================================================

for result_list in parallel_results:
    control_rows.extend(
        result_list
    )


controls_df = pd.DataFrame(
    control_rows
)


# =============================================================================
# 8. COMBINE WITH LINEAR PROBE RESULTS
# =============================================================================

atlas = pd.concat(
    [
        linear_atlas,
        controls_df
    ],
    ignore_index=True
)


# =============================================================================
# 9. SAVE COMPLETE DECODABILITY ATLAS
# =============================================================================

output_file = (
    OUTPUT_DIR
    / "decodability_atlas.csv"
)

atlas.to_csv(
    output_file,
    index=False
)

print(
    f"Saved {len(atlas):,} total results to {output_file}"
)


# =============================================================================
# 10. RESULT COUNTS
# =============================================================================

display(
    atlas.groupby(
        "control"
    ).size().rename(
        "n_results"
    )
)

In [8]:
# # =============================================================================
# # V4.1:
#   *----| ORIGINAL SERIAL CONTROL PIPELINE |--------------------*****
# # Computes majority, lexical TF-IDF, shuffled-label, and random-projection
# # baselines for each task/test split. Hidden-state readout vectors are
# # recomputed inside every layer/readout loop, making this version expensive
# # and highly redundant for large probing experiments.
# # =============================================================================
# control_rows = []
# for task, (label, flag) in TASKS.items():
#     data = probe_examples[probe_examples[flag]].copy(); train = data.query("split == 'dev'")
#     for test_split in eligible_splits(task, data):
#         test = data.query('split == @test_split'); y_train, y_test = train[label].to_numpy(), test[label].to_numpy()
#         majority = train[label].value_counts(normalize=True).max()
#         control_rows.append({'task': task, 'readout': 'majority', 'layer': -1, 'test_split': test_split, 'control': 'majority', 'score': majority})
#         tfidf = TfidfVectorizer(ngram_range=(1, 2)); x_train = tfidf.fit_transform(train.text); x_test = tfidf.transform(test.text)
#         control_rows.append({'task': task, 'readout': 'lexical_tfidf', 'layer': -1, 'test_split': test_split, 'control': 'lexical', 'score': probe_score(x_train, y_train, x_test, y_test)})
#         for readout in READOUTS:
#             for layer in LAYERS:
#                 x_train = np.stack([readout_vector(row, layer, readout) for row in train.itertuples(index=False)]); x_test = np.stack([readout_vector(row, layer, readout) for row in test.itertuples(index=False)])
#                 control_rows.append({'task': task, 'readout': readout, 'layer': layer, 'test_split': test_split, 'control': 'shuffled_labels', 'score': probe_score(x_train, rng.permutation(y_train), x_test, y_test)})
#                 rp = GaussianRandomProjection(n_components=min(128, x_train.shape[1]), random_state=SEED)
#                 control_rows.append({'task': task, 'readout': readout, 'layer': layer, 'test_split': test_split, 'control': 'random_projection', 'score': probe_score(rp.fit_transform(x_train), y_train, rp.transform(x_test), y_test)})
# atlas = pd.concat([linear_atlas, pd.DataFrame(control_rows)], ignore_index=True)
# atlas.to_csv(OUTPUT_DIR / 'decodability_atlas.csv', index=False)
# display(atlas.groupby('control').size().rename('n_results'))


NameError: name 'eligible_splits' is not defined

In [12]:
# # =============================================================================
# # V4.2:
#       PARALLELIZED CONTROL PIPELINE WITH MATRIX CACHING
# #
# # Runs majority and TF-IDF controls serially, then parallelizes the
# # shuffled-label and random-projection controls across readouts and layers.
# # Reuses cached train/test hidden-state matrices when available, but may still
# # recompute readout vectors while building or filling the cache.
# # =============================================================================
# from joblib import Parallel, delayed

# def process_control_combination(task, label, flag, test_split, readout, layer, train_data, test_data, linear_vectors_cache):
#     data = probe_examples[probe_examples[flag]].copy()
#     train = train_data
#     test = test_data

#     y_train, y_test = train[label].to_numpy(), test[label].to_numpy()

#     # Retrieve pre-calculated feature vectors if available
#     cache_key = (task, readout, layer, test_split)
#     if cache_key in linear_vectors_cache:
#         x_train, y_train_filtered, x_test, y_test_filtered = linear_vectors_cache[cache_key]
#     else:
#         # Recalculate if not cached (though ideally linear_vectors_cache should be comprehensive)
#         x_train_vectors = [readout_vector(row, layer, readout) for row in train.itertuples(index=False)]
#         y_train_filtered = [y_val for i, y_val in enumerate(train[label].to_numpy()) if x_train_vectors[i] is not None]
#         x_train_filtered = [vec for vec in x_train_vectors if vec is not None]

#         x_test_vectors = [readout_vector(row, layer, readout) for row in test.itertuples(index=False)]
#         y_test_filtered = [y_val for i, y_val in enumerate(test[label].to_numpy()) if x_test_vectors[i] is not None]
#         x_test_filtered = [vec for vec in x_test_vectors if vec is not None]

#         if not x_train_filtered or not x_test_filtered or len(set(y_train_filtered)) < 2 or len(set(y_test_filtered)) < 2:
#             return [] # Not enough data for this combination

#         x_train = np.stack(x_train_filtered)
#         y_train = np.array(y_train_filtered)
#         x_test = np.stack(x_test_filtered)
#         y_test = np.array(y_test_filtered)
#         linear_vectors_cache[cache_key] = (x_train, y_train, x_test, y_test)


#     results = []

#     # Shuffled labels
#     if x_train.size > 0 and x_test.size > 0 and len(set(y_train_filtered)) > 1 and len(set(y_test_filtered)) > 1:
#         results.append({'task': task, 'readout': readout, 'layer': layer, 'test_split': test_split, 'control': 'shuffled_labels', 'score': probe_score(x_train, rng.permutation(y_train), x_test, y_test)})

#     # Random projection
#     if x_train.size > 0 and x_test.size > 0 and len(set(y_train_filtered)) > 1 and len(set(y_test_filtered)) > 1 and x_train.shape[1] > 0:
#         rp = GaussianRandomProjection(n_components=min(128, x_train.shape[1]), random_state=SEED)
#         results.append({'task': task, 'readout': readout, 'layer': layer, 'test_split': test_split, 'control': 'random_projection', 'score': probe_score(rp.fit_transform(x_train), y_train, rp.transform(x_test), y_test)})

#     return results

# control_rows = []
# # Cache the x_train/x_test for linear probes to reuse in controls where possible
# linear_vectors_cache = {}

# # Calculate non-layer/readout specific controls first
# for task, (label, flag) in TASKS.items():
#     data = probe_examples[probe_examples[flag]].copy()
#     train = data.query("split == 'dev'")
#     for test_split in eligible_splits(task, data):
#         test = data.query('split == @test_split')
#         y_train, y_test = train[label].to_numpy(), test[label].to_numpy()

#         majority = train[label].value_counts(normalize=True).max()
#         control_rows.append({'task': task, 'readout': 'majority', 'layer': -1, 'test_split': test_split, 'control': 'majority', 'score': majority})

#         tfidf = TfidfVectorizer(ngram_range=(1, 2))
#         # Handle cases where TFIDF might not have enough samples or features
#         if len(train.text) > 0 and len(test.text) > 0:
#             try:
#                 x_train_tfidf = tfidf.fit_transform(train.text)
#                 x_test_tfidf = tfidf.transform(test.text)
#                 if x_train_tfidf.shape[1] > 0 and x_test_tfidf.shape[1] > 0 and len(set(y_train)) > 1 and len(set(y_test)) > 1: # Ensure at least 2 classes for probe
#                     control_rows.append({'task': task, 'readout': 'lexical_tfidf', 'layer': -1, 'test_split': test_split, 'control': 'lexical', 'score': probe_score(x_train_tfidf, y_train, x_test_tfidf, y_test)})
#                 else:
#                     control_rows.append({'task': task, 'readout': 'lexical_tfidf', 'layer': -1, 'test_split': test_split, 'control': 'lexical', 'score': np.nan})
#             except ValueError:
#                 control_rows.append({'task': task, 'readout': 'lexical_tfidf', 'layer': -1, 'test_split': test_split, 'control': 'lexical', 'score': np.nan})
#         else:
#              control_rows.append({'task': task, 'readout': 'lexical_tfidf', 'layer': -1, 'test_split': test_split, 'control': 'lexical', 'score': np.nan})


# # Collect linear probe vectors to avoid recomputing
# for row_idx, row in linear_atlas.iterrows():
#     cache_key = (row['task'], row['readout'], row['layer'], row['test_split'])
#     if row['score'] is not np.nan:
#         # We need to re-extract for caching because linear_atlas only stores scores
#         task_data = probe_examples[probe_examples[TASKS[row['task']][1]]].copy()
#         train_data_filtered = task_data.query("split == 'dev'")
#         test_data_filtered = task_data.query('split == @row.test_split')

#         x_train_vectors = [readout_vector(r, row['layer'], row['readout']) for r in train_data_filtered.itertuples(index=False)]
#         y_train_filtered = [y_val for i, y_val in enumerate(train_data_filtered[TASKS[row['task']][0]].to_numpy()) if x_train_vectors[i] is not None]
#         x_train_filtered = [vec for vec in x_train_vectors if vec is not None]

#         x_test_vectors = [readout_vector(r, row['layer'], row['readout']) for r in test_data_filtered.itertuples(index=False)]
#         y_test_filtered = [y_val for i, y_val in enumerate(test_data_filtered[TASKS[row['task']][0]].to_numpy()) if x_test_vectors[i] is not None]
#         x_test_filtered = [vec for vec in x_test_vectors if vec is not None]

#         if x_train_filtered and x_test_filtered and len(set(y_train_filtered)) > 1 and len(set(y_test_filtered)) > 1:
#             linear_vectors_cache[cache_key] = (np.stack(x_train_filtered), np.array(y_train_filtered), np.stack(x_test_filtered), np.array(y_test_filtered))

# # Parallelize the layer/readout specific controls
# control_results = []
# for task, (label, flag) in TASKS.items():
#     data = probe_examples[probe_examples[flag]].copy()
#     train = data.query("split == 'dev'")
#     for test_split in eligible_splits(task, data):
#         test = data.query('split == @test_split')

#         results_for_task_split = Parallel(n_jobs=-1)(delayed(process_control_combination)(
#             task, label, flag, test_split, readout, layer, train, test, linear_vectors_cache
#         ) for readout in READOUTS for layer in LAYERS)
#         for res_list in results_for_task_split:
#             control_results.extend(res_list)

# control_rows.extend(control_results)

# atlas = pd.concat([linear_atlas, pd.DataFrame(control_rows)], ignore_index=True)
# atlas.to_csv(OUTPUT_DIR / 'decodability_atlas.csv', index=False)
# display(atlas.groupby('control').size().rename('n_results'))

NameError: name 'linear_atlas' is not defined

# 5. Results and interpretation
   Create heatmaps with:
   - Rows: readouts
   - Columns: layers
   - Separate panels: situation variables
   - Values: held-out balanced accuracy
Save the atlas figures and a table of each task/readout’s peak score and layer. If results/layer_curves/heldout_layer_curves.csv is available from V4, join it by readout and layer and calculate correlations between V4 separation and V5 decodability.

In [ ]:
# # =============================================================================
# V5.2: DECODABILITY ATLAS VISUALIZATION + SUMMARY ANALYSIS
#
# Visualizes held-out linear-probe accuracy across GPT-2 XL layers/readouts,
# exports peak decodability summaries, and compares decodability with V4
# geometry curves when available.
# =============================================================================

import matplotlib.pyplot as plt


# =============================================================================
# 1. PLOT DECODABILITY HEATMAPS
# =============================================================================

for test_split in ("test_template", "test_entity", "test_both"):

    subset = atlas.query(
        "test_split == @test_split and control == 'linear'"
    )

    if subset.empty:
        print(f"No linear results for {test_split}; skipping.")
        continue

    tasks = subset["task"].unique()

    fig, axes = plt.subplots(
        len(tasks),
        1,
        figsize=(15, 2.2 * len(tasks)),
        sharex=True
    )

    axes = np.atleast_1d(axes)

    for axis, task in zip(axes, tasks):

        matrix = (
            subset
            .query("task == @task")
            .pivot(
                index="readout",
                columns="layer",
                values="score"
            )
            .reindex(READOUTS)
        )

        image = axis.imshow(
            matrix,
            aspect="auto",
            origin="lower",
            vmin=0,
            vmax=1,
            cmap="viridis"
        )

        axis.set(
            yticks=range(len(READOUTS)),
            yticklabels=READOUTS,
            ylabel=task
        )

        # Optional hypothesized transition boundaries
        axis.axvline(23, color="white", linewidth=1)
        axis.axvline(26, color="white", linewidth=1)

    axes[-1].set_xlabel("GPT-2 XL block")

    fig.colorbar(
        image,
        ax=axes,
        label="Held-out balanced accuracy"
    )

    fig.suptitle(
        f"V5 Decodability Atlas: {test_split}",
        y=1.01
    )

    fig.tight_layout()

    output_path = (
        OUTPUT_DIR
        / f"atlas_{test_split}.png"
    )

    fig.savefig(
        output_path,
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()


# =============================================================================
# 2. PEAK DECODABILITY SUMMARY
# =============================================================================

linear = atlas.query(
    "control == 'linear'"
).copy()


def peak_result(frame):

    valid = frame.dropna(
        subset=["score"]
    )

    if valid.empty:
        return pd.Series({
            "peak_score": np.nan,
            "peak_layer": np.nan
        })

    best_idx = valid["score"].idxmax()

    return pd.Series({
        "peak_score": valid.loc[best_idx, "score"],
        "peak_layer": valid.loc[best_idx, "layer"]
    })


summary = (
    linear
    .groupby(
        ["task", "readout", "test_split"],
        group_keys=False
    )
    .apply(peak_result)
    .reset_index()
)


summary.to_csv(
    OUTPUT_DIR / "decodability_summary.csv",
    index=False
)

display(summary)


# =============================================================================
# 3. COMPARE V5 DECODABILITY WITH V4 GEOMETRY
# =============================================================================

v4_path = (
    ROOT
    / "results"
    / "layer_curves"
    / "heldout_layer_curves.csv"
)


if v4_path.exists():

    geometry = (
        pd.read_csv(v4_path)
        .rename(
            columns={"mean": "geometry"}
        )
    )

    joined = linear.merge(
        geometry[
            [
                "readout",
                "layer",
                "geometry"
            ]
        ],
        on=[
            "readout",
            "layer"
        ],
        how="inner"
    )


    def safe_correlation(frame):

        valid = frame[
            ["geometry", "score"]
        ].dropna()

        if len(valid) < 2:
            return np.nan

        return valid["geometry"].corr(
            valid["score"]
        )


    correlations = (
        joined
        .groupby(
            [
                "task",
                "readout",
                "test_split"
            ]
        )
        .apply(safe_correlation)
        .rename("pearson_r")
        .reset_index()
    )


    correlations.to_csv(
        OUTPUT_DIR
        / "geometry_decodability_correlations.csv",
        index=False
    )

    display(
        correlations.round(3)
    )


else:

    print(
        "V4 numeric curves are absent; "
        "atlas saved, correspondence skipped."
    )


# =============================================================================
# INTERPRETATION BOUNDARY
# =============================================================================

print(
    "Interpretation boundary: "
    "V5 measures decodability; "
    "V6 tests causal necessity."
)

In [ ]:
# ## =============================================================================
# # V5.1: DECODABILITY ATLAS VISUALIZATION + SUMMARY ANALYSIS
# #
# # Visualizes held-out linear-probe accuracy across GPT-2 XL layers/readouts,
# # exports peak decodability summaries, and compares decodability with V4
# # geometry curves when available.
# #
# # Outputs:
# #   - atlas_<test_split>.png
# #   - decodability_summary.csv
# #   - geometry_decodability_correlations.csv
# #
# # Interpretation boundary:
# #   V5 measures whether information is decodable from hidden states.
# #   V6 should test whether those representations are causally necessary.
# # =============================================================================
# for test_split in ('test_template', 'test_entity', 'test_both'):
#     subset = atlas.query("test_split == @test_split and control == 'linear'")
#     if subset.empty: continue
#     tasks = subset.task.unique(); fig, axes = plt.subplots(len(tasks), 1, figsize=(15, 2.2 * len(tasks)), sharex=True); axes = np.atleast_1d(axes)
#     for axis, task in zip(axes, tasks):
#         matrix = subset.query('task == @task').pivot(index='readout', columns='layer', values='score').reindex(READOUTS)
#         image = axis.imshow(matrix, aspect='auto', origin='lower', vmin=0, vmax=1, cmap='viridis')
#         axis.set(yticks=range(len(READOUTS)), yticklabels=READOUTS, ylabel=task); axis.axvline(23, color='white'); axis.axvline(26, color='white')
#     axes[-1].set_xlabel('GPT-2 XL block'); fig.colorbar(image, ax=axes, label='Held-out balanced accuracy'); fig.suptitle(f'V5 atlas: {test_split}', y=1.01); fig.tight_layout(); fig.savefig(OUTPUT_DIR / f'atlas_{test_split}.png', dpi=180, bbox_inches='tight'); plt.show()
# summary = atlas.query("control == 'linear'").groupby(['task', 'readout', 'test_split']).score.agg(['max', 'idxmax']).reset_index(); summary.to_csv(OUTPUT_DIR / 'decodability_summary.csv', index=False); display(summary)
# v4_path = ROOT / 'results' / 'layer_curves' / 'heldout_layer_curves.csv'
# if v4_path.exists():
#     geometry = pd.read_csv(v4_path).rename(columns={'mean': 'geometry'}); joined = atlas.query("control == 'linear'").merge(geometry[['readout', 'layer', 'geometry']], on=['readout', 'layer'])
#     correlations = joined.groupby(['task', 'readout', 'test_split']).apply(lambda frame: frame.geometry.corr(frame.score)).rename('pearson_r').reset_index(); correlations.to_csv(OUTPUT_DIR / 'geometry_decodability_correlations.csv', index=False); display(correlations.round(3))
# else: print('V4 numeric curves are absent; atlas saved, correspondence skipped.')
# print('Interpretation boundary: V5 measures decodability; V6 tests causal necessity.')
